In [2]:
# 0) Setup
import os
import pandas as pd
import numpy as np
import textwrap

# project folder
PROJECT_ROOT = r"C:\Users\dell\Desktop\Documents for UK\dissertation_project"

RAW_DIR = os.path.join(PROJECT_ROOT, "data_raw")
SILVER_DIR = os.path.join(PROJECT_ROOT, "data_silver")
GOLD_DIR = os.path.join(PROJECT_ROOT, "data_gold")
DOCS_DIR = os.path.join(PROJECT_ROOT, "docs")

for d in [RAW_DIR, SILVER_DIR, GOLD_DIR, DOCS_DIR]:
    os.makedirs(d, exist_ok=True)

# input CSVs from data_raw 
INPUTS = {
    "customers.csv": os.path.join(RAW_DIR, "customers.csv"),
    "products.csv": os.path.join(RAW_DIR, "products.csv"),
    "orders.csv": os.path.join(RAW_DIR, "orders.csv"),
    "order_items.csv": os.path.join(RAW_DIR, "order_items.csv"),
    "marketing_spend_daily.csv": os.path.join(RAW_DIR, "marketing_spend_daily.csv"),
    "web_sessions_daily.csv": os.path.join(RAW_DIR, "web_sessions_daily.csv"),
}


missing = [path for path in INPUTS.values() if not os.path.exists(path)]
if missing:
    raise FileNotFoundError(
        "These input files are missing in data_raw:\n" + "\n".join(missing)
    )

print("Setup complete. Using input CSVs from:", RAW_DIR)

Setup complete. Using input CSVs from: C:\Users\dell\Desktop\Documents for UK\dissertation_project\data_raw


In [3]:
# 1) Load raw data
import pandas as pd

def read_raw(name: str) -> pd.DataFrame:
    return pd.read_csv(os.path.join(RAW_DIR, name))

customers = read_raw("customers.csv")
products = read_raw("products.csv")
orders = read_raw("orders.csv")
order_items = read_raw("order_items.csv")
mkt = read_raw("marketing_spend_daily.csv")
web = read_raw("web_sessions_daily.csv")

display({
    "customers": customers.shape,
    "products": products.shape,
    "orders": orders.shape,
    "order_items": order_items.shape,
    "marketing_spend_daily": mkt.shape,
    "web_sessions_daily": web.shape,
})

{'customers': (8000, 4),
 'products': (500, 3),
 'orders': (141085, 8),
 'order_items': (369536, 5),
 'marketing_spend_daily': (731, 7),
 'web_sessions_daily': (731, 5)}

In [4]:
# 2) Step 2.1 — Data Audit
def percent_missing(df: pd.DataFrame) -> pd.Series:
    return (df.isna().mean() * 100).round(2)

def key_dup_count(df: pd.DataFrame, key: str) -> int:
    return int(df.duplicated(subset=[key]).sum()) if key in df.columns else 0

def describe_df(df: pd.DataFrame, key: str | None = None) -> str:
    miss = percent_missing(df)
    top_missing = miss[miss > 0].sort_values(ascending=False).head(12)

    parts = [f"- Rows: **{len(df):,}**, Columns: **{df.shape[1]}**"]
    if key and key in df.columns:
        parts.append(f"- Duplicate {key}: **{key_dup_count(df, key):,}**")
        parts.append(f"- Unique {key}: **{df[key].nunique():,}**")
    if len(top_missing) > 0:
        parts.append("- Top missing columns (%):")
        for col, pct in top_missing.items():
            parts.append(f"  - {col}: {pct}%")
    return "\n".join(parts)

# Overview table
overview = pd.DataFrame({
    "file": ["customers.csv","products.csv","orders.csv","order_items.csv","marketing_spend_daily.csv","web_sessions_daily.csv"],
    "rows": [len(customers), len(products), len(orders), len(order_items), len(mkt), len(web)],
    "columns": [customers.shape[1], products.shape[1], orders.shape[1], order_items.shape[1], mkt.shape[1], web.shape[1]],
})
overview.to_csv(os.path.join(DOCS_DIR, "data_audit_overview.csv"), index=False)
overview

,file,rows,columns
0,customers.csv,8000,4
1,products.csv,500,3
2,orders.csv,141085,8
3,order_items.csv,369536,5
4,marketing_spend_daily.csv,731,7
5,web_sessions_daily.csv,731,5


In [5]:
# Date ranges + referential integrity (raw)
orders_dt = pd.to_datetime(orders.get("order_datetime", pd.Series(dtype=str)), errors="coerce")
mkt_dt = pd.to_datetime(mkt.get("date", pd.Series(dtype=str)), errors="coerce")
web_dt = pd.to_datetime(web.get("date", pd.Series(dtype=str)), errors="coerce")

date_ranges = {
    "orders.order_datetime": (orders_dt.min(), orders_dt.max()),
    "marketing_spend_daily.date": (mkt_dt.min(), mkt_dt.max()),
    "web_sessions_daily.date": (web_dt.min(), web_dt.max()),
}
date_ranges

{'orders.order_datetime': (Timestamp('2024-01-01 00:00:00'),
  Timestamp('2025-12-31 00:00:00')),
 'marketing_spend_daily.date': (Timestamp('2024-01-01 00:00:00'),
  Timestamp('2025-12-31 00:00:00')),
 'web_sessions_daily.date': (Timestamp('2024-01-01 00:00:00'),
  Timestamp('2025-12-31 00:00:00'))}

In [6]:
# Referential integrity checks (raw)
missing_customers_raw = orders.loc[~orders["customer_id"].isin(customers["customer_id"]), "customer_id"].nunique()
missing_orders_raw = order_items.loc[~order_items["order_id"].isin(orders["order_id"]), "order_id"].nunique()
missing_products_raw = order_items.loc[~order_items["product_id"].isin(products["product_id"]), "product_id"].nunique()

{
    "orders with customer_id missing in customers": int(missing_customers_raw),
    "order_items with order_id missing in orders": int(missing_orders_raw),
    "order_items with product_id missing in products": int(missing_products_raw),
}

{'orders with customer_id missing in customers': 0,
 'order_items with order_id missing in orders': 0,
 'order_items with product_id missing in products': 0}

In [7]:
# Write audit markdown
audit_sections = []
audit_sections.append("# Phase 2 – Step 2.1 Data Audit Summary\n")
audit_sections.append("## Raw file overview\n")
audit_sections.append("A CSV version of this overview is saved as `docs/data_audit_overview.csv`.\n")

for title, df, key in [
    ("customers.csv", customers, "customer_id"),
    ("products.csv", products, "product_id"),
    ("orders.csv", orders, "order_id"),
    ("order_items.csv", order_items, "order_item_id"),
    ("marketing_spend_daily.csv", mkt, "date"),
    ("web_sessions_daily.csv", web, "date"),
]:
    audit_sections.append(f"## {title}\n{describe_df(df, key)}\n")

audit_sections.append("## Date ranges\n")
audit_sections.append(f"- orders.order_datetime: **{orders_dt.min()} → {orders_dt.max()}**")
audit_sections.append(f"- marketing_spend_daily.date: **{mkt_dt.min()} → {mkt_dt.max()}**")
audit_sections.append(f"- web_sessions_daily.date: **{web_dt.min()} → {web_dt.max()}**\n")

audit_sections.append("## Referential integrity checks (raw)\n")
audit_sections.append(f"- Orders with customer_id not found in customers: **{missing_customers_raw:,}**")
audit_sections.append(f"- Order_items with order_id not found in orders: **{missing_orders_raw:,}**")
audit_sections.append(f"- Order_items with product_id not found in products: **{missing_products_raw:,}**\n")

audit_md_path = os.path.join(DOCS_DIR, "data_audit_summary.md")
with open(audit_md_path, "w", encoding="utf-8") as f:
    f.write("\n".join(audit_sections))

print("Wrote:", audit_md_path)

Wrote: C:\Users\dell\Desktop\Documents for UK\dissertation_project\docs\data_audit_summary.md


In [8]:
# 3) Step 2.2 — Cleaning to Silver layer
customers_s = customers.copy()
products_s = products.copy()
orders_s = orders.copy()
order_items_s = order_items.copy()
mkt_s = mkt.copy()
web_s = web.copy()

# customers
customers_s["signup_date"] = pd.to_datetime(customers_s["signup_date"], errors="coerce").dt.date
customers_s = customers_s.drop_duplicates(subset=["customer_id"])
customers_s["country"] = customers_s["country"].fillna("Unknown")
customers_s["segment"] = customers_s["segment"].fillna("Unknown")

# products
products_s = products_s.drop_duplicates(subset=["product_id"])
products_s["category"] = products_s["category"].fillna("Unknown")
products_s["base_unit_price_gbp"] = pd.to_numeric(products_s["base_unit_price_gbp"], errors="coerce")
products_s = products_s[products_s["base_unit_price_gbp"].fillna(0) > 0]

# orders
orders_s = orders_s.drop_duplicates(subset=["order_id"])
orders_s["order_datetime"] = pd.to_datetime(orders_s["order_datetime"], errors="coerce")
orders_s["order_date"] = orders_s["order_datetime"].dt.date
orders_s["channel"] = orders_s["channel"].fillna("Unknown")
orders_s["discount_rate"] = pd.to_numeric(orders_s["discount_rate"], errors="coerce").fillna(0).clip(0, 1)
orders_s["shipping_days"] = pd.to_numeric(orders_s["shipping_days"], errors="coerce").fillna(1).round().astype(int)
orders_s.loc[orders_s["shipping_days"] < 1, "shipping_days"] = 1
orders_s["order_total_gbp"] = pd.to_numeric(orders_s["order_total_gbp"], errors="coerce").fillna(0)
orders_s.loc[orders_s["order_total_gbp"] < 0, "order_total_gbp"] = 0
orders_s["is_returned"] = orders_s["is_returned"].astype(str).str.lower().isin(["true", "1", "yes", "y"])

# order_items
order_items_s = order_items_s.drop_duplicates()
order_items_s["quantity"] = pd.to_numeric(order_items_s["quantity"], errors="coerce").fillna(1).round().astype(int)
order_items_s.loc[order_items_s["quantity"] < 1, "quantity"] = 1
order_items_s["unit_price_gbp"] = pd.to_numeric(order_items_s["unit_price_gbp"], errors="coerce").fillna(0)
order_items_s = order_items_s[order_items_s["unit_price_gbp"] > 0]
order_items_s["line_total_gbp"] = (order_items_s["quantity"] * order_items_s["unit_price_gbp"]).round(2)

# marketing
mkt_s["date"] = pd.to_datetime(mkt_s["date"], errors="coerce").dt.date
for col in mkt_s.columns:
    if col != "date":
        mkt_s[col] = pd.to_numeric(mkt_s[col], errors="coerce").fillna(0)
        mkt_s.loc[mkt_s[col] < 0, col] = 0

# web
web_s["date"] = pd.to_datetime(web_s["date"], errors="coerce").dt.date
web_s["sessions"] = pd.to_numeric(web_s["sessions"], errors="coerce").fillna(0).round().astype(int)
web_s["orders"] = pd.to_numeric(web_s["orders"], errors="coerce").fillna(0).round().astype(int)
web_s.loc[web_s["sessions"] < 0, "sessions"] = 0
web_s.loc[web_s["orders"] < 0, "orders"] = 0
web_s["conversion_rate_calc"] = np.where(web_s["sessions"] > 0, web_s["orders"] / web_s["sessions"], 0).round(5)

# Referential integrity (silver)
missing_customers = orders_s.loc[~orders_s["customer_id"].isin(customers_s["customer_id"]), "customer_id"].nunique()
missing_orders = order_items_s.loc[~order_items_s["order_id"].isin(orders_s["order_id"]), "order_id"].nunique()
missing_products = order_items_s.loc[~order_items_s["product_id"].isin(products_s["product_id"]), "product_id"].nunique()

print("Silver referential integrity:")
print("missing customers:", int(missing_customers))
print("missing orders:", int(missing_orders))
print("missing products:", int(missing_products))

Silver referential integrity:
missing customers: 0
missing orders: 0
missing products: 0


In [9]:
# Save Silver CSVs + cleaning log
silver_outputs = {
    "customers_silver.csv": customers_s,
    "products_silver.csv": products_s,
    "orders_silver.csv": orders_s,
    "order_items_silver.csv": order_items_s,
    "marketing_spend_daily_silver.csv": mkt_s,
    "web_sessions_daily_silver.csv": web_s,
}
for fn, df in silver_outputs.items():
    df.to_csv(os.path.join(SILVER_DIR, fn), index=False)

cleaning_log_path = os.path.join(DOCS_DIR, "cleaning_log.md")
with open(cleaning_log_path, "w", encoding="utf-8") as f:
    f.write(textwrap.dedent(f"""    # Phase 2 – Step 2.2 Cleaning Log (Silver Layer)

    ## Rules applied
    - Parsed date/datetime columns and created `order_date`.
    - Removed duplicate primary keys (customers, products, orders) and exact duplicate order_item rows.
    - Filled missing categorical fields with `Unknown`.
    - Enforced numeric ranges: discount_rate ∈ [0,1], shipping_days ≥ 1, non-negative spend/sessions/orders/totals.
    - Normalised `is_returned` to boolean.
    - Created `line_total_gbp` in order_items.

    ## Referential integrity after cleaning
    - Orders with customer_id not found in customers: **{missing_customers:,}**
    - Order_items with order_id not found in orders: **{missing_orders:,}**
    - Order_items with product_id not found in products: **{missing_products:,}**
    """))

print("Saved silver outputs to:", SILVER_DIR)
print("Wrote:", cleaning_log_path)

Saved silver outputs to: C:\Users\dell\Desktop\Documents for UK\dissertation_project\data_silver
Wrote: C:\Users\dell\Desktop\Documents for UK\dissertation_project\docs\cleaning_log.md


In [10]:
# 4) Step 2.3 — Star schema docs + DimDate
schema_md = textwrap.dedent("""# Phase 2 – Step 2.3 Data Model (Star Schema)

## Dimensions
### DimDate (generated; also provided as `data_silver/dim_date.csv`)
- Grain: 1 row per day
- Key: `date`
- Attributes: year, quarter, month, month_name, week_start, day_of_week, is_weekend

### DimCustomer (`customers_silver.csv`)
- Key: `customer_id`
- Attributes: signup_date, country, segment

### DimProduct (`products_silver.csv`)
- Key: `product_id`
- Attributes: category, base_unit_price_gbp

## Facts
### FactOrders (`orders_silver.csv`)
- Grain: 1 row per order
- PK: `order_id`
- FKs: `customer_id` → DimCustomer, `order_date` → DimDate
- Attributes: channel, discount_rate, shipping_days, is_returned
- Measures: order_total_gbp

### FactOrderItems (`order_items_silver.csv`)
- Grain: 1 row per order line
- PK: `order_item_id`
- FKs: `order_id` → FactOrders, `product_id` → DimProduct
- Measures: quantity, unit_price_gbp, line_total_gbp

### FactMarketingDaily (`marketing_spend_daily_silver.csv`)
- Grain: 1 row per day
- FK: `date` → DimDate
- Measures: total_spend_gbp and spend-by-channel columns

### FactWebDaily (`web_sessions_daily_silver.csv`)
- Grain: 1 row per day
- FK: `date` → DimDate
- Measures: sessions, orders, conversion_rate_calc, avg_order_value_gbp (if present)

## Relationships (Power BI)
- DimCustomer[customer_id] 1—* FactOrders[customer_id]
- FactOrders[order_id] 1—* FactOrderItems[order_id]
- DimProduct[product_id] 1—* FactOrderItems[product_id]
- DimDate[date] 1—* FactOrders[order_date]
- DimDate[date] 1—* FactMarketingDaily[date]
- DimDate[date] 1—* FactWebDaily[date]
""")

schema_path = os.path.join(DOCS_DIR, "schema.md")
with open(schema_path, "w", encoding="utf-8") as f:
    f.write(schema_md)

# Create DimDate based on union of available dates
all_dates = pd.concat([
    pd.to_datetime(orders_s["order_date"], errors="coerce"),
    pd.to_datetime(mkt_s["date"], errors="coerce"),
    pd.to_datetime(web_s["date"], errors="coerce"),
]).dropna().dt.date
date_min, date_max = min(all_dates), max(all_dates)

dimdate = pd.date_range(date_min, date_max, freq="D")
dim_date = pd.DataFrame({"date": dimdate.date})
dim_date["year"] = dimdate.year
dim_date["quarter"] = dimdate.quarter
dim_date["month"] = dimdate.month
dim_date["month_name"] = dimdate.strftime("%b")
dim_date["day_of_week"] = dimdate.strftime("%a")
dim_date["is_weekend"] = (dimdate.dayofweek >= 5).astype(int)
dim_date["week_start"] = (dimdate - pd.to_timedelta(dimdate.dayofweek, unit="D")).date

dim_date.to_csv(os.path.join(SILVER_DIR, "dim_date.csv"), index=False)

print("Wrote:", schema_path)
print("Wrote:", os.path.join(SILVER_DIR, "dim_date.csv"))
print("DimDate range:", date_min, "to", date_max)

Wrote: C:\Users\dell\Desktop\Documents for UK\dissertation_project\docs\schema.md
Wrote: C:\Users\dell\Desktop\Documents for UK\dissertation_project\data_silver\dim_date.csv
DimDate range: 2024-01-01 to 2025-12-31


In [11]:
# 5) Step 2.4 — Gold KPI tables (for Power BI + Agent)

# Gold daily KPIs
orders_daily = (
    orders_s.groupby("order_date", as_index=False)
    .agg(
        revenue=("order_total_gbp", "sum"),
        orders=("order_id", "count"),
        returned_orders=("is_returned", "sum"),
        avg_shipping_days=("shipping_days", "mean"),
    )
    .rename(columns={"order_date": "date"})
)
orders_daily["aov"] = np.where(orders_daily["orders"] > 0, orders_daily["revenue"] / orders_daily["orders"], 0).round(2)
orders_daily["return_rate"] = np.where(orders_daily["orders"] > 0, orders_daily["returned_orders"] / orders_daily["orders"], 0).round(4)
orders_daily["avg_shipping_days"] = orders_daily["avg_shipping_days"].round(2)

mkt_daily = mkt_s[["date", "total_spend_gbp"]].copy()
web_daily = web_s[["date", "sessions", "orders", "conversion_rate_calc"]].copy().rename(columns={"orders": "web_orders"})

gold_daily = orders_daily.merge(mkt_daily, on="date", how="left").merge(web_daily, on="date", how="left")
gold_daily["total_spend_gbp"] = gold_daily["total_spend_gbp"].fillna(0)
gold_daily["cac_proxy"] = np.where(gold_daily["orders"] > 0, gold_daily["total_spend_gbp"] / gold_daily["orders"], 0).round(2)
gold_daily = gold_daily.sort_values("date")
gold_daily.to_csv(os.path.join(GOLD_DIR, "gold_daily_kpis.csv"), index=False)

# Gold daily channel KPIs
orders_ch = (
    orders_s.groupby(["order_date", "channel"], as_index=False)
    .agg(revenue=("order_total_gbp", "sum"), orders=("order_id", "count"))
    .rename(columns={"order_date": "date"})
)

channel_cols = [c for c in mkt_s.columns if c.endswith("_spend_gbp") and c != "total_spend_gbp"]
mkt_long = mkt_s.melt(id_vars=["date"], value_vars=channel_cols, var_name="channel_col", value_name="spend_channel_gbp")
mkt_long["channel"] = (
    mkt_long["channel_col"]
    .str.replace("_spend_gbp", "", regex=False)
    .str.replace("_", " ", regex=False)
    .str.title()
)
mkt_long = mkt_long[["date", "channel", "spend_channel_gbp"]]

gold_channel = orders_ch.merge(mkt_long, on=["date", "channel"], how="left")
gold_channel["spend_channel_gbp"] = gold_channel["spend_channel_gbp"].fillna(0).round(2)
gold_channel["cac_proxy_channel"] = np.where(gold_channel["orders"] > 0, gold_channel["spend_channel_gbp"] / gold_channel["orders"], 0).round(2)
gold_channel.to_csv(os.path.join(GOLD_DIR, "gold_daily_channel_kpis.csv"), index=False)

# Gold weekly category KPIs
items_with_cat = (
    order_items_s.merge(products_s[["product_id", "category"]], on="product_id", how="left")
    .merge(orders_s[["order_id", "order_date", "is_returned"]], on="order_id", how="left")
)
items_with_cat["order_date"] = pd.to_datetime(items_with_cat["order_date"])
items_with_cat["week_start"] = (
    items_with_cat["order_date"] - pd.to_timedelta(items_with_cat["order_date"].dt.dayofweek, unit="D")
).dt.date

weekly_cat = (
    items_with_cat.groupby(["week_start", "category"], as_index=False)
    .agg(
        revenue=("line_total_gbp", "sum"),
        units_sold=("quantity", "sum"),
        returned_lines=("is_returned", "sum"),
        lines=("order_item_id", "count"),
    )
)
weekly_cat["return_rate"] = np.where(weekly_cat["lines"] > 0, weekly_cat["returned_lines"] / weekly_cat["lines"], 0).round(4)
weekly_cat["revenue"] = weekly_cat["revenue"].round(2)
weekly_cat.to_csv(os.path.join(GOLD_DIR, "gold_weekly_category_kpis.csv"), index=False)

print("Gold outputs written to:", GOLD_DIR)
display({
    "gold_daily_kpis": gold_daily.shape,
    "gold_daily_channel_kpis": gold_channel.shape,
    "gold_weekly_category_kpis": weekly_cat.shape,
})

Gold outputs written to: C:\Users\dell\Desktop\Documents for UK\dissertation_project\data_gold


{'gold_daily_kpis': (731, 12),
 'gold_daily_channel_kpis': (3655, 6),
 'gold_weekly_category_kpis': (840, 7)}

In [12]:
# 6) (Optional) Package outputs into a ZIP for submission/sharing
import zipfile

zip_path = os.path.join(PROJECT_ROOT, "phase2_outputs.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for folder in [RAW_DIR, SILVER_DIR, GOLD_DIR, DOCS_DIR]:
        for root, _, files in os.walk(folder):
            for fn in files:
                full = os.path.join(root, fn)
                arc = os.path.relpath(full, PROJECT_ROOT)
                z.write(full, arcname=arc)

print("Created:", zip_path)

Created: C:\Users\dell\Desktop\Documents for UK\dissertation_project\phase2_outputs.zip
